# C_01 Evaluation Template and Dummy Baseline

**Owner:** C

## Purpose
1. Provide one shared evaluation module (`src/evaluation.py`) so that every model reports
   the same metrics and saves results in the same format.
2. Confirm that every model uses A's fixed cross-validation folds (`data/splits/cv_folds.csv`).
3. Run the Dummy Classifier as the minimum baseline and produce the example output files.

## Agreed protocol
- Features: the 24 features in A's v3 preprocessing template.
- Main comparison: shared random 80/20 split (A's `train_ids.csv` / `test_ids.csv`), threshold 0.5.
- Validation: 5-fold out-of-fold predictions on the random training set, using A's fixed folds.
- Chronological test: refit the same settings on arrivals up to 2017-04-30, test on May-Aug 2017 arrivals; never used for tuning.
- Metrics: Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, Brier score, confusion matrix.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.append(str(project_root / "src"))

import evaluation as ev

warnings.filterwarnings("ignore", category=UserWarning)  # sklearn minor-version pickle warning
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Load A's shared data package

In [2]:
data = ev.load_shared_data(project_root)

for name in ["random_train", "random_test", "chrono_train", "chrono_test"]:
    part = data[name]
    print(f"{name:13s} rows={len(part):6d}  cancellation rate={part['is_canceled'].mean():.4f}")

print("\nModel input features:", len(data["features"]))

random_train  rows= 94851  cancellation rate=0.3726
random_test   rows= 23713  cancellation rate=0.3726
chrono_train  rows= 96428  cancellation rate=0.3648
chrono_test   rows= 22136  cancellation rate=0.4064

Model input features: 24


## 2. Cross-validation folds (A's `cv_folds.csv`)

A fixed the five training-only folds in `data/splits/cv_folds.csv` (numbered 1-5).
`ev.get_cv()` (StratifiedKFold, 5 folds, shuffle, `random_state=42`) reproduces exactly these folds;
`ev.load_shared_data()` checks this row by row and stops with an error if they ever differ.

In [3]:
cv_folds = ev.get_cv_folds(data)   # A's cv_folds.csv
fold_check = cv_folds.merge(data["random_train"][["row_id", "is_canceled"]], on="row_id")
print(fold_check.groupby("cv_fold")["is_canceled"].agg(["size", "mean"]).round(4))
print("\nev.get_cv() matches cv_folds.csv for all", len(cv_folds), "training records.")

          size    mean
cv_fold               
1        18971  0.3726
2        18970  0.3726
3        18970  0.3726
4        18970  0.3726
5        18970  0.3726

ev.get_cv() matches cv_folds.csv for all 94851 training records.


## 3. Dummy Classifier (minimum baseline)

`strategy="prior"` always predicts the majority class ("not cancelled") — identical to
`most_frequent` at the 0.5 threshold — but its predicted probability equals the training
cancellation rate. This gives a meaningful reference for the Brier score and calibration.
No tuning is applied.

In [4]:
dummy_params = {"strategy": "prior"}
dummy_pipe = ev.make_pipeline(data, DummyClassifier(**dummy_params))

dummy_metrics, dummy_preds = ev.run_standard_experiment(
    dummy_pipe, model="dummy", data=data, params=dummy_params, verbose=False
)
display(dummy_metrics.round(4))

   model variant     experiment      subset  threshold      n  n_positive  accuracy  precision  recall   f1  roc_auc  pr_auc   brier     tn  fp     fn  tp
0  dummy    main         random  validation        0.5  94851       35341    0.6274        0.0     0.0  0.0      0.5  0.3726  0.2338  59510   0  35341   0
1  dummy    main         random        test        0.5  23713        8835    0.6274        0.0     0.0  0.0      0.5  0.3726  0.2338  14878   0   8835   0
2  dummy    main  chronological        test        0.5  22136        8997    0.5936        0.0     0.0  0.0      0.5  0.4064  0.2430  13139   0   8997   0


**Reading the baseline.** Accuracy is about 0.63 on the random test set and 0.59 on the chronological test (May–Aug 2017 arrivals, where 40.6% of bookings were cancelled), yet recall and F1 are 0 because no cancellation is identified. ROC-AUC is 0.5 and PR-AUC equals the cancellation rate. Any useful model must clearly beat these values on recall, F1, ROC-AUC and PR-AUC — accuracy alone is not evidence of a useful model.

## 4. Output formats (example files for D and E)

In [5]:
print("Metrics table columns:", ev.METRIC_COLUMNS)
print("Prediction table columns:", ev.PREDICTION_COLUMNS)
display(dummy_preds.groupby(["experiment", "subset"]).head(2))

Metrics table columns: ['model', 'variant', 'experiment', 'subset', 'threshold', 'n', 'n_positive', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'brier', 'tn', 'fp', 'fn', 'tp']
Prediction table columns: ['row_id', 'model', 'variant', 'experiment', 'subset', 'fold', 'hotel', 'y_true', 'y_prob']
        row_id  model variant     experiment      subset  fold         hotel  y_true    y_prob
0            2  dummy    main         random  validation     1  Resort Hotel       0  0.372588
1            3  dummy    main         random  validation     4  Resort Hotel       0  0.372597
94851        4  dummy    main         random        test    -1  Resort Hotel       0  0.372595
94852        7  dummy    main         random        test    -1  Resort Hotel       0  0.372595
118564   11288  dummy    main  chronological        test    -1  Resort Hotel       1  0.364821
118565   11289  dummy    main  chronological        test    -1  Resort Hotel       1  0.364821


In [6]:
print(dummy_preds.groupby(["experiment", "subset"]).size())

experiment     subset    
chronological  test          22136
random         test          23713
               validation    94851
dtype: int64


## 5. Checks and helper functions

- `ev.validate_outputs(...)` must pass before results are handed to C.
- `ev.metrics_from_predictions(preds, threshold=t)` recomputes all metrics at any threshold
  from the prediction table (used by E for the threshold analysis).

In [7]:
ev.validate_outputs(dummy_metrics, dummy_preds, data)
print("Output check passed.")

display(ev.metrics_from_predictions(dummy_preds, threshold=0.3).round(4))

Output check passed.
   model variant     experiment      subset  threshold      n  n_positive  accuracy  precision  recall      f1  roc_auc  pr_auc   brier  tn     fp  fn     tp
0  dummy    main         random  validation        0.3  94851       35341    0.3726     0.3726     1.0  0.5429      0.5  0.3726  0.2338   0  59510   0  35341
1  dummy    main         random        test        0.3  23713        8835    0.3726     0.3726     1.0  0.5429      0.5  0.3726  0.2338   0  14878   0   8835
2  dummy    main  chronological        test        0.3  22136        8997    0.4064     0.4064     1.0  0.5780      0.5  0.4064  0.2430   0  13139   0   8997


## 6. Files produced

| File | Content |
|---|---|
| `results/metrics/dummy_main_metrics.csv` | Dummy metrics (validation, random test, chronological test) |
| `results/predictions/dummy_main_predictions.csv` | Dummy record-level probabilities |
| `results/models/dummy_main.joblib` / `.json` | Fitted pipeline and settings |